# Harmonize two VCFs from different pipelines

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fbcouto/multimerge-genomics-research/blob/main/notebooks/files/harmonize_two_vcfs.ipynb)

**The problem.** Two VCFs made by different pipelines rarely line up. One names chromosomes `chr1`, the
other `1`; one declares the length of each sequence, the other only the name of the genome build; one
writes two alternative alleles in one record, the other in two. A naive join (same chromosome and
position) then finds nothing in common, or worse, silently pairs coordinates of different builds. Here
the two files are the Genome in a Bottle calls of one person (HG002) and the 1000 Genomes Project
frequency file: *which of this person's variants are known in the population, and at what frequency?*

**What you need.** Two VCFs (plain or `.gz`, any size) of the same species.

**What you get.** Both files validated, on one chromosome naming, checked for the same reference
build, normalized (multiallelic records split, alleles trimmed), sorted, BGZF-compressed and indexed
(`.tbi`, ready for `tabix`, `bcftools` and IGV); the counts of shared positions and variants; and a
table of the variants of A found in B, with B's allele frequency.

In [ ]:
# Setup. Runs on Google Colab or on your own computer: on Colab, the first run installs MMGEN from PyPI
# (about a minute). Tables use pandas and plots matplotlib when installed (Colab has both); without
# them the notebook prints plain text and skips the plots.
import platform, subprocess, sys
from pathlib import Path

RELEASE = "https://github.com/fbcouto/multimerge-genomics-research/releases/download/v2.5.0"      # the demo packs
PACKAGE = "multimerge-genomics==2.5.0"   # from PyPI
try:
    import multimerge_genomics as mmg
except ImportError:
    if platform.system() != "Linux":
        print("MMGEN is not installed in this Python. Install it with:\n"
              "  python -m pip install multimerge-genomics\n"
              "and run this cell again.")
        raise
    print("Installing MMGEN (about a minute)...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", PACKAGE], check=True)
    import multimerge_genomics as mmg

IN_NOTEBOOK = "ipykernel" in sys.modules
try:
    from IPython.display import display, Markdown
except ImportError:                        # plain Python: print instead
    display, Markdown = print, str
try:
    import pandas as pd
    pd.set_option("display.width", 160)
    pd.set_option("display.max_columns", 30)
except ImportError:
    pd = None
try:
    import matplotlib
    if not IN_NOTEBOOK:
        matplotlib.use("Agg")              # plain Python: save the figures, open no window
    import matplotlib.pyplot as plt
except ImportError:
    plt = None

# Chart colors (a palette checked for color-blind readers): two series, a two-sided scale, neutral ink
BLUE, ORANGE, RED, GRAY, INK = "#2a78d6", "#eb6834", "#e34948", "#a3a29c", "#52514e"


def show(rows, n=25):
    """Shows a list of dictionaries as a table (pandas when available, plain text otherwise)."""
    rows = list(rows)
    if pd is not None:
        display(pd.DataFrame(rows[:n]))
    else:
        for row in rows[:n]:
            print("  ".join(f"{k}: {v}" for k, v in row.items()))
    if len(rows) > n:
        print(f"... and {len(rows) - n} more rows")


def tidy(ax):
    """Recessive axes: no top or right frame, a light grid behind the data."""
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(color="#e4e3df", linewidth=0.8)
    ax.set_axisbelow(True)


def save_figure(fig, path):
    """Saves a figure; shows it in a notebook, closes it in plain Python."""
    fig.savefig(path, dpi=150, bbox_inches="tight")
    print("Figure saved to", path)
    if IN_NOTEBOOK:
        plt.show()
    else:
        plt.close(fig)


print(mmg.engine_info())

## Data

By default, two public files from the human demo pack (downloaded once, about 45 MB):

* **A**: the Genome in a Bottle (NIST) benchmark calls v4.2.1 of HG002, a public genome of a volunteer
  who consented to open sharing, cut to chromosome 22 and some genes. VCF 4.2, names `chr1`, contig
  lengths declared.
* **B**: the 1000 Genomes Project GRCh38 sites release (2019), cut to the sites of the demo genomes.
  VCF 4.3, names `1`, no contig lengths, allele frequencies in `INFO` (rounded to 2 decimals).

To use your own files, set `MY_VCF_A` and `MY_VCF_B`.

In [ ]:
# Your files: set the paths here (None = the demo data)
MY_VCF_A = None              # e.g. "my_sample.vcf.gz"
MY_VCF_B = None              # e.g. "other_pipeline.vcf"
AF_FIELD = "AF"              # the INFO field of B to report for the shared variants
REFERENCE_FASTA = None       # the genome FASTA of the build: also left-aligns indels (recommended)

import urllib.request, zipfile

DATA = Path("data")                          # the demo files are extracted here
OUT = Path("results") / "harmonize_two_vcfs"         # everything this notebook writes
OUT.mkdir(parents=True, exist_ok=True)


def demo_file(pack, member):
    """One file of a demo pack: downloads the pack once into the working folder and extracts only that file."""
    target = DATA / member
    if not target.exists():
        if not Path(pack).exists():
            print(f"Downloading {pack} from the MMGEN release...")
            urllib.request.urlretrieve(f"{RELEASE}/{pack}", pack + ".part")
            Path(pack + ".part").replace(pack)
        with zipfile.ZipFile(pack) as z:
            z.extract(member, DATA)
    return str(target)


USING_DEMO = MY_VCF_A is None and MY_VCF_B is None
VCF_A = MY_VCF_A or demo_file("mmgen_demo_human_v3.zip", "human/giab/HG002.vcf.gz")
VCF_B = MY_VCF_B or demo_file("mmgen_demo_human_v3.zip", "human/af_1000g_demo.vcf.gz")
print("A:", VCF_A)
print("B:", VCF_B)

## Step 1. Validate both files

A file that is not what it claims (a truncated download, a missing column, a position that is not a
number) makes every later step fail with an obscure message, or silently drop records. The validator
reads the whole file and reports each kind of problem once, with the line where it first appears and how
to fix it. It also says whether the file is sorted and how it names chromosomes.

In [ ]:
report = {}
for name, path in (("A", VCF_A), ("B", VCF_B)):
    v = mmg.validate_file(path)
    report[name] = v
    print(f"{name}: valid {v['valid']}, {v['records']:,} records, {v['errors']} error(s), {v['warnings']} warning(s), "
          f"sorted {v['sorted']}, chromosome names '{v['naming']}'")
    for p in v["problems"][:5]:
        print(f"   line {p['line']}: {p['message']}" + (f"  -> {p['fix']}" if p["fix"] else ""))
if not all(v["valid"] for v in report.values()):
    raise RuntimeError("Fix the errors above first: each one comes with its line and how to fix it.")

## Step 2. One chromosome naming, one reference build

UCSC names chromosomes `chr1 ... chrX, chrM`; Ensembl and NCBI write `1 ... X, MT`. Tools that join files
by name find zero matches between the two conventions, usually without any error. B is rewritten in A's
convention (records and `##contig` header lines); a file already in that convention is left as it is.

Position 1,000,000 of chromosome 1 is a different base in GRCh37 and in GRCh38: matching positions across
builds is meaningless. MMGEN compares the sequence lengths each header declares
(`##contig=<ID=chr1,length=248956422>`), matching `chr1` with `1`; when a file declares no lengths, it
falls back on the build named in the header. If the builds differ, lift one file over first (CrossMap,
Picard LiftoverVcf or UCSC liftOver); MMGEN does not lift over.

In [ ]:
style = report["A"]["naming"]
if style not in ("chr", "no_chr"):
    raise RuntimeError(f"A names its chromosomes '{style}': rename them first, with a mapping file "
                       "(see help(mmg.normalize_chromosomes)).")
renamed = {}
for name, path in (("A", VCF_A), ("B", VCF_B)):
    if report[name]["naming"] == style:
        renamed[name] = path
        print(f"{name}: already uses '{style}' names")
    else:
        renamed[name] = str(OUT / f"{name}.renamed.vcf.gz")
        r = mmg.normalize_chromosomes(path, renamed[name], style=style)
        print(f"{name}: {r['changed']:,} of {r['records']:,} records renamed, for example {r['converted'][:3]}; "
              f"names left as they were: {r['kept'] or 'none'}")

for name in ("A", "B"):
    ref = mmg.file_reference(renamed[name])
    print(f"{name}: build {ref['assembly']}; declared: {ref['declared'] or '-'}; "
          f"{len(ref['sequences'])} sequence lengths in the header")
same = mmg.compare_references(renamed["A"], renamed["B"])
print("\nVerdict:", same["verdict"])
print(f"   sequences with the same length: {same['matching']}, with different lengths: {len(same['different'])}")
if same["compatible"] is False:
    raise RuntimeError("Different reference builds: lift one file over to the other's build before comparing.")
if same["compatible"] is None:
    print("   The headers do not say enough: confirm the build in the files' documentation before going on.")
elif same["matching"] == 0:
    print("   (no lengths to compare in one of the files: the verdict rests on the build names they declare)")

## Step 3. One way of writing each variant; sorted and indexed

The same variant can be written in several ways: two alternative alleles in one record (`C -> CCGGT,CCTGT`)
or one record per allele; with extra shared bases (`CA -> TA` is `C -> T`); an indel in a repeat at its
left or its right end. `vcf_normalize` splits multiallelic records (one ALT per record, genotypes
rewritten) and trims shared bases. Moving indels to the left end of a repeat needs the genome
(`REFERENCE_FASTA`); without it, some indels written differently by the two pipelines will not match.

Both files are sorted in the same chromosome order (natural: `chr2` before `chr10`), written as BGZF (an
ordinary gzip made of blocks) and indexed (`.tbi`). The index lets any tool jump to a region without
reading the whole file. Sorting works for files larger than the memory: what does not fit is sorted in
runs on disk and merged.

In [ ]:
normalized = {}
for name in ("A", "B"):
    normalized[name] = str(OUT / f"{name}.norm.vcf.gz")
    r = mmg.vcf_normalize(renamed[name], normalized[name], split_multiallelic=True, reference=REFERENCE_FASTA)
    print(f"{name}: {r['read']:,} records -> {r['written']:,}; {r['split']:,} multiallelic split, "
          f"{r['trimmed']:,} trimmed, {r['left_aligned']:,} left-aligned")

ready = {}
for name in ("A", "B"):
    ready[name] = str(OUT / f"{name}.sorted.vcf.gz")
    records, runs, skipped = mmg.sort_genomic([normalized[name]], ready[name], chrom_order="natural", index=True)
    print(f"{name}: {records:,} records, {runs} run(s) on disk, {skipped} line(s) skipped -> {ready[name]} (+ .tbi)")

## Step 4. Compare

Two records are the same variant when chromosome, position, REF and ALT are all equal; a shared position
with other alleles is a different variant (or the same one written differently). The files are read one
chromosome at a time through the index, so the memory needed is that of one chromosome, not of the
whole genome.

In [ ]:
def by_chromosome(path, chrom):
    """{(chrom, pos, ref, alt): INFO} of one chromosome, read through the index."""
    records = {}
    for line in mmg.query_region(path, chrom, header=False):
        f = line.split("\t", 8)
        records[(f[0], int(f[1]), f[3], f[4])] = f[7]
    return records


def info_value(info, key):
    for item in info.split(";"):
        if item.startswith(key + "="):
            return item[len(key) + 1:]
    return None


chromosomes = sorted({c for name in ("A", "B") for c, _ in mmg.vcf_stats(ready[name])["sequences"]})
n_a = n_b = shared_positions = shared = 0
found = {"SNVs": [0, 0], "indels and others": [0, 0]}      # [in A, also in B]
rows = []
for chrom in chromosomes:
    a, b = by_chromosome(ready["A"], chrom), by_chromosome(ready["B"], chrom)
    n_a, n_b = n_a + len(a), n_b + len(b)
    shared_positions += len({k[:2] for k in a} & {k[:2] for k in b})
    for key in a:
        kind = "SNVs" if len(key[2]) == len(key[3]) == 1 else "indels and others"
        found[kind][0] += 1
        if key in b:
            found[kind][1] += 1
            shared += 1
            rows.append({"chrom": key[0], "pos": key[1], "ref": key[2], "alt": key[3], AF_FIELD: info_value(b[key], AF_FIELD)})

print(f"A: {n_a:,} variants; B: {n_b:,} variants")
print(f"Shared positions: {shared_positions:,}; shared variants (same REF and ALT): {shared:,} ({shared / max(n_a, 1):.1%} of A)")
for kind, (in_a, in_b) in found.items():
    print(f"   {kind} of A found in B: {in_b:,} of {in_a:,} ({in_b / max(in_a, 1):.1%})")
print(f"Only in A: {n_a - shared:,}; only in B: {n_b - shared:,}")

with open(OUT / "shared_with_af.tsv", "w") as f:
    f.write("chrom\tpos\tref\talt\t" + AF_FIELD + "\n")
    for r in rows:
        f.write(f"{r['chrom']}\t{r['pos']}\t{r['ref']}\t{r['alt']}\t{r[AF_FIELD]}\n")
print("Table written to", OUT / "shared_with_af.tsv")

freqs = [float(r[AF_FIELD]) for r in rows if r[AF_FIELD] not in (None, ".")]
if freqs:
    print(f"\nAmong the shared variants with a value of {AF_FIELD}: {sum(x >= 0.05 for x in freqs):,} common (>= 5%), "
          f"{sum(0.01 <= x < 0.05 for x in freqs):,} low-frequency (1-5%), {sum(x < 0.01 for x in freqs):,} rare (< 1%)")
show(rows[:10])

## Check the result

Things to look at before trusting the comparison. `OK` lines passed; a `CHECK` line deserves a look.
With the demo data, the expected numbers are also asserted.

In [ ]:
def check(condition, text):
    print(("OK     " if condition else "CHECK  ") + text)


for name in ("A", "B"):
    v = mmg.validate_file(ready[name])
    check(v["valid"] and v["sorted"], f"{name}: the final file is valid and sorted")
    check(Path(ready[name] + ".tbi").exists(), f"{name}: its index (.tbi) is there")
    check(v["naming"] == style, f"{name}: chromosome names are '{style}'")
check(shared <= min(n_a, n_b), "shared variants do not outnumber either file")
snv_rate = found["SNVs"][1] / max(found["SNVs"][0], 1)
indel_rate = found["indels and others"][1] / max(found["indels and others"][0], 1)
check(snv_rate >= indel_rate, f"SNVs match at least as often as indels ({snv_rate:.1%} against {indel_rate:.1%}); "
      "if not, the indels may be written differently: give REFERENCE_FASTA")
if USING_DEMO:
    assert snv_rate > 0.8, "most SNVs of HG002 should be in the 1000 Genomes file"
    assert shared_positions >= shared > 0
    print("Demo expectations met.")

## Methods paragraph

A starting text for the methods section of a paper or report, filled in with the version and the parameters
this run used. Edit it to your study, and cite the methods MMGEN reproduces (see References).

In [ ]:
import textwrap
VERSION = mmg.engine_info()["version"]
methods = f"""Variant files were checked with validate_file, chromosome names were converted to the "{style}" convention
(normalize_chromosomes) and the reference builds were compared from what the headers declare, sequence lengths or
build name (compare_references: {same['verdict'].rstrip('.')}). Multiallelic records were split and alleles trimmed to their shortest
representation with vcf_normalize ({'with left-alignment of indels against ' + str(REFERENCE_FASTA) if REFERENCE_FASTA else 'without left-alignment of indels'}),
and the files were sorted in natural chromosome order, BGZF-compressed and tabix-indexed (sort_genomic, index=True).
Variants were matched on chromosome, position, REF and ALT. All steps used MMGEN version {VERSION}
(Couto F. MMGEN: MultiMerge Genomics (version {VERSION}). Zenodo. https://doi.org/10.5281/zenodo.20417492)."""
print(textwrap.fill(" ".join(methods.split()), 110))

## Adapt it to your data

* Give `REFERENCE_FASTA` (the genome of the build, plain or `.gz`) to left-align indels: indel matches go up.
* Comparing two callers on the same sample? Swap step 6's question: the variants only in A and only in B are
  where the callers disagree; restrict both files to the regions both can call first (`vcf_filter(regions=...)`).
* For a whole genome, nothing changes: every step streams, and step 6 holds one chromosome at a time.
* Different builds (step 3 fails): lift one file over with CrossMap or Picard LiftoverVcf, then run this again.
* Keep the sorted, indexed outputs: `query_region` (or `tabix`) answers region questions on them instantly.

## References

* Danecek P. et al. (2011) The variant call format and VCFtools. *Bioinformatics* 27:2156.
* Li H. (2011) Tabix: fast retrieval of sequence features from generic TAB-delimited files. *Bioinformatics* 27:718.
* Tan A., Abecasis G.R., Kang H.M. (2015) Unified representation of genetic variants. *Bioinformatics* 31:2202.
* Wagner J. et al. (2022) Benchmarking challenging small variants with linked and long reads. *Cell Genomics* 2:100128 (GIAB v4.2.1).
* Lowy-Gallego E. et al. (2019) Variant calling on the GRCh38 assembly with the data from phase three of the
  1000 Genomes Project. *Wellcome Open Research* 4:50.
* Zhao H. et al. (2014) CrossMap: a versatile tool for coordinate conversion between genome assemblies. *Bioinformatics* 30:1006.